## Análise de custos - Bronze Layer

In [0]:
from pyspark.sql import functions as F

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"

df_custos = spark.table(f"{bronze_schema}.custos")
total = df_custos.count()
display(df_custos)

### Funções auxiliares

In [0]:
TOKENS_NULOS = ["NULL", "NONE", "NAN", "N/A"]      # textos que na prática significam "sem valor"

def texto(c):
    """Coluna como string, sem espaços nas pontas."""
    return F.trim(F.col(c).cast("string"))

def eh_faltante(c):
    """Null de verdade, string vazia/só espaços, ou texto tipo 'NULL'."""
    return F.col(c).isNull() | (texto(c) == "") | F.upper(texto(c)).isin(TOKENS_NULOS)

def mostrar(df_, descricao):
    """Imprime a descrição e faz o display. Se o DataFrame estiver vazio, informa que não há casos."""
    if df_.isEmpty():
        print(f"\n{descricao}: nenhum caso encontrado")
    else:
        print(f"\n{descricao}")
        display(df_)

def custo_numerico(c):
    """Converte o texto de custo para double só para análise: tira 'R$', 'reais' e espaços, troca vírgula por ponto."""
    limpo = F.regexp_replace(texto(c), r"R\$|reais|\s", "")
    return F.regexp_replace(limpo, ",", ".").cast("double")

### 1. Duplicidade

In [0]:
colunas_negocio = [c for c in df_custos.columns if c != "ingestion_datetime"]
distintas = df_custos.select(*colunas_negocio).distinct().count()
print(f"Linhas excedentes por cópia exata: {total - distintas}")

for chave in ["id_custo", "id_chamado"]:
    outras = [c for c in colunas_negocio if c != chave]
    repetidos = (
        df_custos.filter(~eh_faltante(chave))
          .groupBy(texto(chave).alias(chave))
          .agg(
              F.count("*").alias("ocorrencias"),
              F.size(F.collect_set(F.struct(*outras))).alias("versoes_distintas"),
          )
          .filter(F.col("ocorrencias") > 1)
    )
    n_ids = repetidos.count()
    linhas_envolvidas = repetidos.agg(F.sum("ocorrencias")).first()[0] or 0
    n_divergentes = repetidos.filter(F.col("versoes_distintas") > 1).count()
    print(f"\n{chave} repetidos: {n_ids} ids em {linhas_envolvidas} linhas ({linhas_envolvidas - n_ids} linhas excedentes)")
    print(f"  -> {n_divergentes} desses ids aparecem com conteúdo diferente entre as linhas")
    mostrar(repetidos.orderBy(F.desc("versoes_distintas"), F.desc("ocorrencias"), chave),
            f"{chave} repetidos (ocorrencias = nº de linhas; versoes_distintas = nº de conteúdos diferentes)")
    if chave == "id_chamado":
        ids_chamado_repetidos = repetidos

In [0]:
# linhas completas dos id_chamado com mais de um custo (join sobre o id com o mesmo trim usado no agrupamento)
linhas_repetidas = (
    df_custos
      .withColumn("id_chave", texto("id_chamado"))
      .join(ids_chamado_repetidos.select(F.col("id_chamado").alias("id_chave")), on="id_chave", how="inner")
      .drop("id_chave")
      .orderBy("id_chamado", "id_custo")
)
mostrar(linhas_repetidas, "Linhas completas dos id_chamado com mais de um custo")

### 2. Entradas faltantes por coluna

In [0]:
exprs = []
for c in df_custos.columns:
    exprs += [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}__nulo"),
        F.sum(F.when(texto(c) == "", 1).otherwise(0)).alias(f"{c}__vazio"),
        F.sum(F.when(F.upper(texto(c)).isin(TOKENS_NULOS), 1).otherwise(0)).alias(f"{c}__texto_nulo"),
    ]
r = df_custos.agg(*exprs).first()

df_custos_faltantes = (
    spark.createDataFrame(
        [(c, r[f"{c}__nulo"], r[f"{c}__vazio"], r[f"{c}__texto_nulo"]) for c in df_custos.columns],
        "coluna string, nulo long, vazio long, texto_nulo long",
    )
    .withColumn("total_faltante", F.col("nulo") + F.col("vazio") + F.col("texto_nulo"))
    .withColumn("pct_faltante", F.round(F.col("total_faltante") * 100 / total, 2))
    .orderBy(F.desc("pct_faltante"))
)
mostrar(df_custos_faltantes,
        "Faltantes por coluna: null real, string vazia e texto tipo 'NULL', com % sobre o total de linhas")

### 3. Formato de custo

In [0]:
v = texto("custo")
df_custos_fmt = (
    df_custos
      .withColumn(
          "formato",
          F.when(eh_faltante("custo"), "faltante")
           .when(v.rlike(r"^R\$\s*-?\d+[.,]\d+$"), "com R$")
           .when(v.rlike(r"^-?\d+[.,]\d+\s*reais$"), "com 'reais'")
           .when(v.rlike(r"^-?\d+,\d+$"), "vírgula decimal")
           .when(v.rlike(r"^-?\d+\.\d+$"), "ponto decimal")
           .when(v.rlike(r"^-?\d+$"), "inteiro")
           .otherwise("não reconhecido"),
      )
      .withColumn("negativo", v.contains("-"))
)
mostrar(
    df_custos_fmt.groupBy("formato").agg(
          F.count("*").alias("count"),
          F.sum(F.col("negativo").cast("int")).alias("negativos"),
      )
      .withColumn("pct", F.round(F.col("count") * 100 / total, 2))
      .orderBy(F.desc("count")),
    "Distribuição de custo por formato (negativos = quantos daquele formato têm sinal de menos)",
)
mostrar(
    df_custos_fmt.groupBy("formato").agg(F.slice(F.collect_set("custo"), 1, 5).alias("exemplos")),
    "Até 5 exemplos de cada formato",
)
mostrar(
    df_custos_fmt.filter(F.col("formato") == "não reconhecido").groupBy("custo").count().orderBy(F.desc("count")),
    "Valores de custo em formato não reconhecido",
)

### 4. Valores de custo
Conversão feita só para a análise. A tabela Bronze não é alterada.

In [0]:
df_custos_num = (df_custos.filter(~eh_faltante("custo"))
                   .withColumn("custo_num", custo_numerico("custo")))

n_nao_convertidos = df_custos_num.filter(F.col("custo_num").isNull()).count()
print(f"Valores que não converteram para número: {n_nao_convertidos}")

mostrar(
    df_custos_num.agg(
        F.min("custo_num").alias("minimo"),
        F.max("custo_num").alias("maximo"),
        F.round(F.avg("custo_num"), 2).alias("media"),
        F.percentile_approx("custo_num", 0.5).alias("mediana"),
        F.sum(F.when(F.col("custo_num") < 0, 1).otherwise(0)).alias("negativos"),
        F.sum(F.when(F.col("custo_num") == 0, 1).otherwise(0)).alias("zeros"),
    ),
    "Resumo dos valores de custo convertidos",
)
mostrar(df_custos_num.orderBy(F.desc("custo_num")).limit(10), "10 maiores custos")
mostrar(df_custos_num.filter(F.col("custo_num") < 0).orderBy("custo_num"), "Custos negativos")

### 5. Integridade com chamados

In [0]:
ids_chamados = (spark.table(f"{bronze_schema}.chamados")
                  .filter(~eh_faltante("id_chamado"))
                  .select(texto("id_chamado").alias("id_chamado")).distinct())
ids_custos = (df_custos.filter(~eh_faltante("id_chamado"))
                .select(texto("id_chamado").alias("id_chamado")).distinct())

custos_sem_chamado = ids_custos.join(ids_chamados, on="id_chamado", how="left_anti")
chamados_sem_custo = ids_chamados.join(ids_custos, on="id_chamado", how="left_anti")

print(f"id_chamado em custos que não existem em chamados: {custos_sem_chamado.count()}")
print(f"Chamados sem nenhum custo: {chamados_sem_custo.count()}")
mostrar(custos_sem_chamado.orderBy("id_chamado"), "id_chamado de custos sem chamado correspondente")
mostrar(chamados_sem_custo.orderBy("id_chamado"), "Chamados sem custo")

### Conclusões e tratamentos previstos para a Silver

| Problema encontrado | Evidência | Tratamento na Silver |
|---|---|---|
| Cópias exatas | 781 linhas idênticas, sem versões divergentes | `dropDuplicates` |
| `custo` em 4 formatos | vírgula decimal (78.533), ponto decimal (26.256), com "R$" (15.676), com "reais" (10.604) | Remover "R$", "reais" e espaços, trocar vírgula por ponto e converter para `decimal(10,2)` |
| Custos negativos | 271 valores após remover duplicatas (mínimo -100,95) | Converter para null |
| Custos zerados | 273 valores após remover duplicatas | Manter |
| Faltantes | Nenhum nulo, vazio ou texto tipo `NULL` | Nenhum |
| Integridade com chamados | Todos os `id_chamado` de custos existem em chamados. 1.550 chamados (1,18%) não têm custo | Manter os chamados sem custo. Nas métricas da Gold, tratar como custo não informado, sem contar como zero |